# TCC - Análise da Produção e Faturamento de Mel no Brasil

## 1. Introdução

**Autor**: Gabriel de Jesus.

O mel, um produto natural de grande valor nutritivo e econômico, desempenha um papel significativo na economia agrícola brasileira. A apicultura, atividade responsável pela sua produção, contribui para a renda de milhares de famílias e é vital para a polinização de diversas culturas agrícolas.

Este trabalho tem como objetivo principal analisar os fatores que afetam a produção e o faturamento anual de mel no Brasil. Para tanto, realiza-se a exploração de uma base de dados abrangente, combinando informações sobre a produção e o valor do mel com dados climáticos para identificar padrões e tendências.

É importante notar que, para garantir a consistência e a possibilidade de cruzamento das informações, a análise se concentrará nos dados do período de 2000 a 2024, intervalo de tempo em que os conjuntos de dados de produção de mel e de observações climáticas se sobrepõem.

Para mais detalhes sobre as fontes de dados utilizadas e suas especificidades, recomenda-se a consulta ao arquivo README.md na raiz deste repositório.



## 2. Preparação e Limpeza dos Dados

#### Bibliotecas utilizadas

In [35]:
import pandas as pd
pd.options.display.float_format = '{:.2f}'.format
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#### Funções Auxiliares

In [36]:
def show_head_tail(df):
    display(df.head(5))
    display(df.tail(5))

In [37]:
def show_max(df, column):
    print("Máximo valor em", column)
    display(df[df[column] == df[column].max()])
    
def show_min(df, column):
    print("Mínimo valor em", column)
    display(df[df[column] == df[column].min()])

In [38]:
def listar_valores_unicos(df, coluna):
    print(f"\nValores únicos na coluna '{coluna}':")
    print(df[coluna].unique())
    print(df[coluna].value_counts())    

In [ ]:
def multiplicar_colunas_numericas(df,valor):    
    resultado = df.copy()
    for coluna in resultado.columns:
        if resultado[coluna].dtype == 'float64' or resultado[coluna].dtype == 'int64':
            resultado[coluna] = resultado[coluna] * valor
    return resultado

### Brasil

#### Análise Temporal do Faturamento Nacional de Mel

##### Extração

In [39]:
faturamento_br_bruto = pd.read_csv('data/faturamento mel/Faturamento R$ - Mel de Abelha - Brasil - 2000 2024.csv')
faturamento_br_bruto

,Sigla,Código,Brasil,2000,2001,2002,2003,2004,2005,2006,...,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
0,BR,0,Brasil,191235.45,179290.16,205953.34,269407.15,267114.75,243996.10,253066.58,...,246313.20,299400.12,314682.62,294926.53,278263.99,334416.94,397418.25,422375.01,370795.28,397857.63


O conjunto de dados original apresenta uma estrutura horizontal, o que limita a eficiência das ferramentas de análise. Portanto, realizou-se o tratamento desses dados para o formato vertical (melt), facilitando a manipulação e visualização.

In [40]:
melted_faturamento_br = pd.melt(faturamento_br_bruto.drop(columns=['Sigla', 'Código']), id_vars=['Brasil'], var_name='Ano', value_name='Faturamento (R$)')
show_head_tail(melted_faturamento_br)

,Brasil,Ano,Faturamento (R$)
0,Brasil,2000,191235.45
1,Brasil,2001,179290.16
2,Brasil,2002,205953.34
3,Brasil,2003,269407.15
4,Brasil,2004,267114.75


,Brasil,Ano,Faturamento (R$)
20,Brasil,2020,334416.94
21,Brasil,2021,397418.25
22,Brasil,2022,422375.01
23,Brasil,2023,370795.28
24,Brasil,2024,397857.63


Observa-se que os dados originais estão expressos em milhares de reais. Por esse motivo, os valores foram convertidos para a unidade monetária real, multiplicando-os por 1.000.

In [ ]:
faturamento_br = multiplicar_colunas_numericas(melted_faturamento_br, 1000)
show_head_tail(faturamento_br)
#191235448.330

,Brasil,Ano,Faturamento (R$)
0,Brasil,2000,191235448.33
1,Brasil,2001,179290160.27
2,Brasil,2002,205953340.51
3,Brasil,2003,269407150.09
4,Brasil,2004,267114750.17


,Brasil,Ano,Faturamento (R$)
20,Brasil,2020,334416942.33
21,Brasil,2021,397418252.93
22,Brasil,2022,422375012.79
23,Brasil,2023,370795279.61
24,Brasil,2024,397857627.91


In [43]:
faturamento_br.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Brasil            25 non-null     object 
 1   Ano               25 non-null     object 
 2   Faturamento (R$)  25 non-null     float64
dtypes: float64(1), object(2)
memory usage: 732.0+ bytes


##### Análise do Faturamento Nominal

In [44]:
show_max(faturamento_br, 'Faturamento (R$)')
show_min(faturamento_br, 'Faturamento (R$)')

Máximo valor em Faturamento (R$)


,Brasil,Ano,Faturamento (R$)
22,Brasil,2022,422375012.79


Mínimo valor em Faturamento (R$)


,Brasil,Ano,Faturamento (R$)
1,Brasil,2001,179290160.27


Com base nos dados analisados, 2022 registrou o maior faturamento nacional com a venda de mel. Contudo, os valores apresentados encontram-se deflacionados para o ano de 2010, segundo o IPCA (Índice Nacional de Preços ao Consumidor Amplo). Isto significa que estão ajustados com base no poder de compra do consumidor do ano de 2010. Para uma análise atualizada, será realizada uma nova base de índice de preços para o ano de 2024, por se tratar do período mais recente contemplado nos dados.

O IPCA constitui o principal indicador de inflação no Brasil. Este índice mensura a variação de preços de um conjunto representativo de produtos e serviços, permitindo o ajuste de valores monetários ao longo do tempo para refletir o poder de compra real da moeda. A desconsideração da inflação resultaria em comparações imprecisas entre valores nominais de diferentes épocas.